### Decision Tree Model

In [1]:
""" 
This notebook implements a decision tree model
with target GMM weights , input : axle loading
"""

' \nThis notebook implements a decision tree model\nwith target GMM weights , input : axle loading\n'

In [1]:
from pyspark.sql import SparkSession

spark = SparkSession.builder \
    .appName("Capstone_Data") \
    .master("local[*]") \
    .getOrCreate()

In [2]:
#historical dataset 2019 - 2023
df = spark.read.csv("Data\Raw_Data\Weigh_in_Motion_19_23.csv", header=True, inferSchema=True)

<>:2: SyntaxWarning: invalid escape sequence '\R'
<>:2: SyntaxWarning: invalid escape sequence '\R'
C:\Users\rubie\AppData\Local\Temp\ipykernel_18232\3529866632.py:2: SyntaxWarning: invalid escape sequence '\R'
  df = spark.read.csv("Data\Raw_Data\Weigh_in_Motion_19_23.csv", header=True, inferSchema=True)


In [3]:
df2 = spark.read.csv("Data\Raw_Data\Weigh_in_Motion_24_25.csv", header=True, inferSchema=True)

<>:1: SyntaxWarning: invalid escape sequence '\R'
<>:1: SyntaxWarning: invalid escape sequence '\R'
C:\Users\rubie\AppData\Local\Temp\ipykernel_18232\3059976666.py:1: SyntaxWarning: invalid escape sequence '\R'
  df2 = spark.read.csv("Data\Raw_Data\Weigh_in_Motion_24_25.csv", header=True, inferSchema=True)


In [4]:
#aggregating both datasets into 1
df3 = df.union(df2)

In [5]:
#Creating a date column
from pyspark.sql.functions import col, concat, lpad
df4 = df3.withColumn(
    "date",
    concat(
        col("year"),
        lpad(col("month"), 2, "0"),  
        lpad(col("day"), 2, "0")
    )
)


In [6]:
#Filtering for QueensBound Data and casting gvw column as float type
from pyspark.sql import functions as F

df5 = df4.filter("direction = 'QB'").withColumn("gvw", F.col("gvw").cast("float"))

In [7]:
df5 = df5.fillna(0)

In [8]:
# Removing outliers *asssume pandas df, targets gvw col
#Outliers are removed using vehicle gvw 
def remove_outliers_iqr(df, factor):
    Q1 = df["gvw"].quantile(0.25)
    Q3 = df["gvw"].quantile(0.75)
    IQR = Q3 - Q1
    lower_bound = Q1 - factor * IQR
    upper_bound = Q3 + factor * IQR

    outliers_iqr = df[(df["gvw"] < lower_bound) | (df["gvw"] > upper_bound)]
    
    print(f"Outlier Count: {len(outliers_iqr)}")
    print(f"Original Count: {len(df)}")
    print(f"Remaining count: {len(df) - len(outliers_iqr)}")
    print(f"Outlier percentage: {len(outliers_iqr)/len(df)*100:.2f}%")
    print(f"Upper bound: {upper_bound:.2f}, Lower bound: {lower_bound:.2f}")
    
    # Return filtered dataframe
    return df[(df["gvw"] >= lower_bound) & (df["gvw"] <= upper_bound)], upper_bound, lower_bound

In [9]:
#eliminate outliers from spark df
#Outliers are removed using vehicle gvw 
from pyspark.sql import functions as F

def remove_outliers_iqr2(df_spark, factor):
    # Step 1: Get Q1 and Q3
    q1, q3 = df_spark.approxQuantile("gvw", [0.25, 0.75], 0.01)

    # Step 2: Compute IQR
    iqr = q3 - q1

    # Step 3: Define bounds
    lower_bound = q1 - factor * iqr
    upper_bound = q3 + factor * iqr

    print(f"Upper bound: {upper_bound}, Lower bound: {lower_bound}")

    # Step 4: Filter outliers
    return df_spark.filter((F.col("gvw") >= lower_bound) & (F.col("gvw") <= upper_bound)) , upper_bound, lower_bound

In [10]:
from pyspark.ml.feature import VectorAssembler
from pyspark.ml.clustering import GaussianMixture as spark_GM

def fit_gmm_spark(df_spark, cols, num_clusters, max_iter):
    
    assembler = VectorAssembler(inputCols=cols, outputCol="features")
    df_input = assembler.transform(df_spark)

    gmm = spark_GM(k=num_clusters, maxIter=max_iter)

    # Fit the model
    model = gmm.fit(df_input)

    #weights for each record
    df_spark2 = model.transform(df_input)

    return model, df_spark2


In [11]:
#format the "probability" col for df_spark2
from pyspark.sql.functions import col
from pyspark.ml.functions import vector_to_array
from pyspark.sql import functions as F
from pyspark.sql.types import StringType

def format_dt_inputs(df_spark2):
    # Extract probability vector into individual cluster columns
    df_spark3 = (df_spark2
        .withColumn("probability_array", vector_to_array(col("probability")))
        .withColumn("cluster1", col("probability_array")[0])
        .withColumn("cluster2", col("probability_array")[1])
        .withColumn("cluster3", col("probability_array")[2])
        .drop("probability_array", "probability")  # Can drop multiple columns at once
    )

    # Parse date and extract temporal features
    df_spark3 = (df_spark3
        .withColumn("parsed_date", F.to_date(F.col("date").cast(StringType()), "yyMMdd"))
        .withColumn("week_of_year", F.weekofyear("parsed_date"))
        .withColumn("day_of_week", F.dayofweek("parsed_date") - 1)  # Convert to 0-6 range (Monday=0)
        .drop("parsed_date")
    )

    return df_spark3

In [12]:
from sklearn.metrics import mean_squared_error, r2_score
from sklearn.model_selection import GridSearchCV

#Excepts a pandas df/samples
def dt_grid_search(X_train, y_train, X_test, y_test, model, param_grid, cv=5):    
    grid_search = GridSearchCV(
        estimator=model,
        param_grid=param_grid,
        cv=cv,
        scoring='neg_mean_squared_error',  # Use negative MSE for scikit-learn convention
        n_jobs=-1,
        return_train_score=True,
        verbose=1
    )
    
    grid_search.fit(X_train, y_train)
    
    # Calculate additional metrics
    best_model = grid_search.best_estimator_
    y_pred = best_model.predict(X_test)
    
    mse = mean_squared_error(y_test, y_pred)
    r2 = r2_score(y_test, y_pred)
    
    print("="*50)
    print("REGRESSION GRID SEARCH RESULTS")
    print("="*50)
    print(f"Best parameters: {grid_search.best_params_}")
    print(f"Best CV Score (-MSE): {grid_search.best_score_}")
    print(f"Test MSE: {mse}")
    
    return grid_search

In [13]:
def predict_cluster_weights(input_data, models):
    k1_wt = models[0].predict(input_data)[0]
    k2_wt = models[1].predict(input_data)[0]
    k3_wt = models[2].predict(input_data)[0]

    return [k1_wt, k2_wt, k3_wt]


In [14]:
pc = 5 # percision for values in regression tree visuals

Class 2

In [15]:
import pandas as pd
#filtering by class
class_num = 2
sample_ratio = .01
cols = [f"wt{i}" for i in range(1, 6)] #Max number of axles for this class is 5, determined in kmeans notebook

df_spark = df5.filter(f"class = {class_num}")

iqr_factor = 1.6
df_spark = remove_outliers_iqr2(df_spark, iqr_factor)[0]

Upper bound: 6664.0, Lower bound: 196.0


In [16]:
max_iter = 20
num_clusters = 4
k3_model, df_spark2 = fit_gmm_spark(df_spark,cols, num_clusters, max_iter)

In [17]:
# Get the weights
weights = k3_model.weights

# Print them nicely
print("GMM Component Weights:")
for i, weight in enumerate(weights):
    print(f"Component {i}: {weight:.4f} ({weight * 100:.2f}%)")

GMM Component Weights:
Component 0: 0.9960 (99.60%)
Component 1: 0.0001 (0.01%)
Component 2: 0.0001 (0.01%)
Component 3: 0.0038 (0.38%)


In [ ]:
df_spark3 = format_dt_inputs(df_spark2)

In [ ]:
#Sample for parameter tuning
df_sample = df_spark3.sample(
    withReplacement=False,  
    fraction=sample_ratio,                          
).toPandas()

In [ ]:
from sklearn.model_selection import train_test_split

input_cols = ["week_of_year", "day_of_week", "hour"]
output_cols = ["cluster1", "cluster2", "cluster3"]

df_sample[input_cols] = df_sample[input_cols].astype('category')
X_train, X_test, y_train, y_test = train_test_split(df_sample[input_cols], df_sample[output_cols], test_size=0.3)

print(f"Training set size: {X_train.shape[0]}")
print(f"Test set size: {X_test.shape[0]}")


Training set size: 438703
Test set size: 188016


In [ ]:
import numpy as np
min_sample_leaf_array = ((np.arange(0.01, .11, .01))*X_train.shape[0]).astype("int")
min_sample_leaf_array = np.unique(min_sample_leaf_array)
max_depth_array = np.arange(10, 101, 10)

param_grid = {
    'min_samples_leaf' : min_sample_leaf_array,
    'max_depth' : max_depth_array
}

In [ ]:
from sklearn.tree import DecisionTreeRegressor
from sklearn import tree
import matplotlib.pyplot as plt

dt_model = DecisionTreeRegressor()
input_cols = ["week_of_year", "day_of_week", "hour"]
output = "cluster1"

grid_search = dt_grid_search(X_train, y_train[output], X_test, y_test[output], dt_model, param_grid)

best_model = grid_search.best_estimator_
model_k1 = best_model

Fitting 5 folds for each of 100 candidates, totalling 500 fits
REGRESSION GRID SEARCH RESULTS
Best parameters: {'max_depth': np.int64(10), 'min_samples_leaf': np.int64(43870)}
Best CV Score (-MSE): -4.0012747878714106e-05
Test MSE: 3.743604643566734e-05


In [ ]:
from sklearn.tree import export_graphviz
import graphviz

dot_data = export_graphviz(best_model, 
                           out_file=None,
                           feature_names=input_cols,
                           filled=True,
                           rounded=True,
                           special_characters=True,
                           precision=pc)

graph = graphviz.Source(dot_data)
graph.render(f"./decision_tree_vis/class_{class_num}/cluster1", format='pdf', cleanup=True)  # Creates PDF
# graph  # Display in notebook

'decision_tree_vis\\class_2\\cluster1.pdf'

In [ ]:
from sklearn.tree import DecisionTreeRegressor
from sklearn import tree
import matplotlib.pyplot as plt

dt_model = DecisionTreeRegressor()
input_cols = ["week_of_year", "day_of_week", "hour"]
output = "cluster2"

grid_search = dt_grid_search(X_train, y_train[output], X_test, y_test[output], dt_model, param_grid)

best_model = grid_search.best_estimator_
model_k2 = best_model

Fitting 5 folds for each of 100 candidates, totalling 500 fits
REGRESSION GRID SEARCH RESULTS
Best parameters: {'max_depth': np.int64(20), 'min_samples_leaf': np.int64(4387)}
Best CV Score (-MSE): -0.003397865404374439
Test MSE: 0.003295255126360605


In [ ]:
from sklearn.tree import export_graphviz
import graphviz

dot_data = export_graphviz(best_model, 
                           out_file=None,
                           feature_names=input_cols,
                           filled=True,
                           rounded=True,
                           special_characters=True,
                           precision=pc)

graph = graphviz.Source(dot_data)
graph.render(f"./decision_tree_vis/class_{class_num}/cluster2", format='pdf', cleanup=True)  # Creates PDF
# graph  # Display in notebook

'decision_tree_vis\\class_2\\cluster2.pdf'

In [ ]:
from sklearn.tree import DecisionTreeRegressor
from sklearn import tree
import matplotlib.pyplot as plt

dt_model = DecisionTreeRegressor()
input_cols = ["week_of_year", "day_of_week", "hour"]
output = "cluster3"

grid_search = dt_grid_search(X_train, y_train[output], X_test, y_test[output], dt_model, param_grid)

best_model = grid_search.best_estimator_
model_k3 = best_model

Fitting 5 folds for each of 100 candidates, totalling 500 fits
REGRESSION GRID SEARCH RESULTS
Best parameters: {'max_depth': np.int64(10), 'min_samples_leaf': np.int64(4387)}
Best CV Score (-MSE): -0.0036581116794155944
Test MSE: 0.003534647412712553


In [28]:
from sklearn.tree import export_graphviz
import graphviz

dot_data = export_graphviz(best_model, 
                           out_file=None,
                           feature_names=input_cols,
                           filled=True,
                           rounded=True,
                           special_characters=True,
                           precision=pc)

graph = graphviz.Source(dot_data)
graph.render(f"./decision_tree_vis/class_{class_num}/cluster3", format='pdf', cleanup=True)  # Creates PDF
# graph  # Display in notebook

'decision_tree_vis\\class_2\\cluster3.pdf'

In [29]:
#No error handling implemented yet, care with args passed
week_of_year = 51
day_of_week = 3
hour = 10

#arg vars are above this pd df is for organizing for model passing
input_data = pd.DataFrame({
    'week_of_year': [week_of_year],
    'day_of_week': [day_of_week], 
    'hour': [hour]
})

models = [model_k1, model_k2, model_k3]

wts = predict_cluster_weights(input_data, models)

print(f"Inputs\nWeek of year : {week_of_year}\nDay of week: {day_of_week}\nHour: {hour}")
for i in range(0, 3):
    print(f"Cluster {i+1} predicted weight: {wts[i]:.3f} , {wts[i]*100:.3f}%")

Inputs
Week of year : 51
Day of week: 3
Hour: 10
Cluster 1 predicted weight: 0.000 , 0.020%
Cluster 2 predicted weight: 0.009 , 0.894%
Cluster 3 predicted weight: 0.991 , 99.147%


Class 3

In [30]:
import pandas as pd
#filtering by class
class_num = 3
sample_ratio = .01
cols = [f"wt{i}" for i in range(1, 6)] #Max number of axles for this class is 5, determined in kmeans notebook

df_spark = df5.filter(f"class = {class_num}")

iqr_factor = 2.5
df_spark = remove_outliers_iqr2(df_spark, iqr_factor)[0]

Upper bound: 12790.0, Lower bound: -1970.0


In [31]:
max_iter = 20
num_clusters = 3
k3_model, df_spark2 = fit_gmm_spark(df_spark,cols, num_clusters, max_iter)

In [32]:
# Get the weights
weights = k3_model.weights

# Print them nicely
print("GMM Component Weights:")
for i, weight in enumerate(weights):
    print(f"Component {i}: {weight:.4f} ({weight * 100:.2f}%)")

GMM Component Weights:
Component 0: 0.9751 (97.51%)
Component 1: 0.0124 (1.24%)
Component 2: 0.0124 (1.24%)


In [33]:
df_spark3 = format_dt_inputs(df_spark2)

In [34]:
#Sample for parameter tuning
df_sample = df_spark3.sample(
    withReplacement=False,  
    fraction=sample_ratio,                          
).toPandas()

In [35]:
from sklearn.model_selection import train_test_split

input_cols = ["week_of_year", "day_of_week", "hour"]
output_cols = ["cluster1", "cluster2", "cluster3"]

df_sample[input_cols] = df_sample[input_cols].astype('category')
X_train, X_test, y_train, y_test = train_test_split(df_sample[input_cols], df_sample[output_cols], test_size=0.3)

print(f"Training set size: {X_train.shape[0]}")
print(f"Test set size: {X_test.shape[0]}")


Training set size: 70326
Test set size: 30141


In [36]:
import numpy as np
min_sample_leaf_array = ((np.arange(0.01, .11, .01))*X_train.shape[0]).astype("int")
min_sample_leaf_array = np.unique(min_sample_leaf_array)
max_depth_array = np.arange(10, 101, 10)

param_grid = {
    'min_samples_leaf' : min_sample_leaf_array,
    'max_depth' : max_depth_array
}

In [37]:
from sklearn.tree import DecisionTreeRegressor
from sklearn import tree
import matplotlib.pyplot as plt

dt_model = DecisionTreeRegressor()
input_cols = ["week_of_year", "day_of_week", "hour"]
output = "cluster1"

grid_search = dt_grid_search(X_train, y_train[output], X_test, y_test[output], dt_model, param_grid)

best_model = grid_search.best_estimator_
model_k1 = best_model

Fitting 5 folds for each of 100 candidates, totalling 500 fits
REGRESSION GRID SEARCH RESULTS
Best parameters: {'max_depth': np.int64(100), 'min_samples_leaf': np.int64(2109)}
Best CV Score (-MSE): -0.010936119615034668
Test MSE: 0.011273507658841872


In [38]:
from sklearn.tree import export_graphviz
import graphviz

dot_data = export_graphviz(best_model, 
                           out_file=None,
                           feature_names=input_cols,
                           filled=True,
                           rounded=True,
                           special_characters=True,
                           precision=pc)

graph = graphviz.Source(dot_data)
graph.render(f"./decision_tree_vis/class_{class_num}/cluster1", format='pdf', cleanup=True)  # Creates PDF
# graph  # Display in notebook

'decision_tree_vis\\class_3\\cluster1.pdf'

In [39]:
from sklearn.tree import DecisionTreeRegressor
from sklearn import tree
import matplotlib.pyplot as plt

dt_model = DecisionTreeRegressor()
input_cols = ["week_of_year", "day_of_week", "hour"]
output = "cluster2"

grid_search = dt_grid_search(X_train, y_train[output], X_test, y_test[output], dt_model, param_grid)

best_model = grid_search.best_estimator_
model_k2 = best_model

Fitting 5 folds for each of 100 candidates, totalling 500 fits
REGRESSION GRID SEARCH RESULTS
Best parameters: {'max_depth': np.int64(10), 'min_samples_leaf': np.int64(2109)}
Best CV Score (-MSE): -0.0027340299037586675
Test MSE: 0.0028183769147104684


In [40]:
from sklearn.tree import export_graphviz
import graphviz

dot_data = export_graphviz(best_model, 
                           out_file=None,
                           feature_names=input_cols,
                           filled=True,
                           rounded=True,
                           special_characters=True,
                           precision=pc)

graph = graphviz.Source(dot_data)
graph.render(f"./decision_tree_vis/class_{class_num}/cluster2", format='pdf', cleanup=True)  # Creates PDF
# graph  # Display in notebook

'decision_tree_vis\\class_3\\cluster2.pdf'

In [41]:
from sklearn.tree import DecisionTreeRegressor
from sklearn import tree
import matplotlib.pyplot as plt

dt_model = DecisionTreeRegressor()
input_cols = ["week_of_year", "day_of_week", "hour"]
output = "cluster3"

grid_search = dt_grid_search(X_train, y_train[output], X_test, y_test[output], dt_model, param_grid)

best_model = grid_search.best_estimator_
model_k3 = best_model

Fitting 5 folds for each of 100 candidates, totalling 500 fits
REGRESSION GRID SEARCH RESULTS
Best parameters: {'max_depth': np.int64(10), 'min_samples_leaf': np.int64(2109)}
Best CV Score (-MSE): -0.0027340299037586675
Test MSE: 0.0028183769147104684


In [42]:
from sklearn.tree import export_graphviz
import graphviz

dot_data = export_graphviz(best_model, 
                           out_file=None,
                           feature_names=input_cols,
                           filled=True,
                           rounded=True,
                           special_characters=True,
                           precision=pc)

graph = graphviz.Source(dot_data)
graph.render(f"./decision_tree_vis/class_{class_num}/cluster3", format='pdf', cleanup=True)  # Creates PDF
# graph  # Display in notebook

'decision_tree_vis\\class_3\\cluster3.pdf'

In [43]:
#No error handling implemented yet care with args passed
week_of_year = 51
day_of_week = 3
hour = 10

input_data = pd.DataFrame({
    'week_of_year': [week_of_year],
    'day_of_week': [day_of_week], 
    'hour': [hour]
})

models = [model_k1, model_k2, model_k3]

wts = predict_cluster_weights(input_data, models)

print(f"Inputs\nWeek of year : {week_of_year}\nDay of week: {day_of_week}\nHour: {hour}")
for i in range(0, 3):
    print(f"Cluster {i+1} predicted weight: {wts[i]:.3f} , {wts[i]*100:.3f}%")

Inputs
Week of year : 51
Day of week: 3
Hour: 10
Cluster 1 predicted weight: 0.976 , 97.582%
Cluster 2 predicted weight: 0.012 , 1.209%
Cluster 3 predicted weight: 0.012 , 1.209%


Class 4

In [18]:
import pandas as pd
#filtering by class
class_num = 4
sample_ratio = 1.0
cols = [f"wt{i}" for i in range(1, 6)] #Max number of axles for this class is 5, determined in kmeans notebook

df_spark = df5.filter(f"class = {class_num}")

iqr_factor = 1.3
df_spark = remove_outliers_iqr2(df_spark, iqr_factor)[0]

Upper bound: 63208.0, Lower bound: -9368.0


In [19]:
max_iter = 20
num_clusters = 4
k3_model, df_spark2 = fit_gmm_spark(df_spark,cols, num_clusters, max_iter)

In [20]:
# Get the weights
weights = k3_model.weights

# Print them nicely
print("GMM Component Weights:")
for i, weight in enumerate(weights):
    print(f"Component {i}: {weight:.4f} ({weight * 100:.2f}%)")

GMM Component Weights:
Component 0: 0.2500 (25.00%)
Component 1: 0.2500 (25.00%)
Component 2: 0.2500 (25.00%)
Component 3: 0.2500 (25.00%)


In [47]:
df_spark3 = format_dt_inputs(df_spark2)

In [48]:
#Sample for parameter tuning
df_sample = df_spark3.sample(
    withReplacement=False,  
    fraction=sample_ratio,                          
).toPandas()

In [49]:
from sklearn.model_selection import train_test_split

input_cols = ["week_of_year", "day_of_week", "hour"]
output_cols = ["cluster1", "cluster2", "cluster3"]

df_sample[input_cols] = df_sample[input_cols].astype('category')
X_train, X_test, y_train, y_test = train_test_split(df_sample[input_cols], df_sample[output_cols], test_size=0.3)

print(f"Training set size: {X_train.shape[0]}")
print(f"Test set size: {X_test.shape[0]}")


Training set size: 296580
Test set size: 127107


In [50]:
import numpy as np
min_sample_leaf_array = ((np.arange(0.01, .11, .01))*X_train.shape[0]).astype("int")
min_sample_leaf_array = np.unique(min_sample_leaf_array)
max_depth_array = np.arange(10, 101, 10)

param_grid = {
    'min_samples_leaf' : min_sample_leaf_array,
    'max_depth' : max_depth_array
}

In [51]:
from sklearn.tree import DecisionTreeRegressor
from sklearn import tree
import matplotlib.pyplot as plt

dt_model = DecisionTreeRegressor()
input_cols = ["week_of_year", "day_of_week", "hour"]
output = "cluster1"

grid_search = dt_grid_search(X_train, y_train[output], X_test, y_test[output], dt_model, param_grid)

best_model = grid_search.best_estimator_
model_k1 = best_model

Fitting 5 folds for each of 100 candidates, totalling 500 fits
REGRESSION GRID SEARCH RESULTS
Best parameters: {'max_depth': np.int64(10), 'min_samples_leaf': np.int64(2965)}
Best CV Score (-MSE): -5.133369736045246e-26
Test MSE: 6.592454760081269e-25


In [52]:
from sklearn.tree import export_graphviz
import graphviz

dot_data = export_graphviz(best_model, 
                           out_file=None,
                           feature_names=input_cols,
                           filled=True,
                           rounded=True,
                           special_characters=True,
                           precision=pc)

graph = graphviz.Source(dot_data)
graph.render(f"./decision_tree_vis/class_{class_num}/cluster1", format='pdf', cleanup=True)  # Creates PDF
# graph  # Display in notebook

'decision_tree_vis\\class_4\\cluster1.pdf'

In [53]:
from sklearn.tree import DecisionTreeRegressor
from sklearn import tree
import matplotlib.pyplot as plt

dt_model = DecisionTreeRegressor()
input_cols = ["week_of_year", "day_of_week", "hour"]
output = "cluster2"

grid_search = dt_grid_search(X_train, y_train[output], X_test, y_test[output], dt_model, param_grid)

best_model = grid_search.best_estimator_
model_k2 = best_model

Fitting 5 folds for each of 100 candidates, totalling 500 fits
REGRESSION GRID SEARCH RESULTS
Best parameters: {'max_depth': np.int64(10), 'min_samples_leaf': np.int64(2965)}
Best CV Score (-MSE): -2.9967792766084537e-27
Test MSE: 8.652721753019897e-25


In [54]:
from sklearn.tree import export_graphviz
import graphviz

dot_data = export_graphviz(best_model, 
                           out_file=None,
                           feature_names=input_cols,
                           filled=True,
                           rounded=True,
                           special_characters=True,
                           precision=pc)

graph = graphviz.Source(dot_data)
graph.render(f"./decision_tree_vis/class_{class_num}/cluster2", format='pdf', cleanup=True)  # Creates PDF
# graph  # Display in notebook

'decision_tree_vis\\class_4\\cluster2.pdf'

In [55]:
from sklearn.tree import DecisionTreeRegressor
from sklearn import tree
import matplotlib.pyplot as plt

dt_model = DecisionTreeRegressor()
input_cols = ["week_of_year", "day_of_week", "hour"]
output = "cluster3"

grid_search = dt_grid_search(X_train, y_train[output], X_test, y_test[output], dt_model, param_grid)

best_model = grid_search.best_estimator_
model_k3 = best_model

Fitting 5 folds for each of 100 candidates, totalling 500 fits
REGRESSION GRID SEARCH RESULTS
Best parameters: {'max_depth': np.int64(90), 'min_samples_leaf': np.int64(29657)}
Best CV Score (-MSE): -6.738650282834419e-26
Test MSE: 2.725132497674672e-25


In [56]:
from sklearn.tree import export_graphviz
import graphviz

dot_data = export_graphviz(best_model, 
                           out_file=None,
                           feature_names=input_cols,
                           filled=True,
                           rounded=True,
                           special_characters=True,
                           precision=pc)

graph = graphviz.Source(dot_data)
graph.render(f"./decision_tree_vis/class_{class_num}/cluster3", format='pdf', cleanup=True)  # Creates PDF
# graph  # Display in notebook

'decision_tree_vis\\class_4\\cluster3.pdf'

In [57]:
#No error handling implemented yet care with args passed
week_of_year = 51
day_of_week = 3
hour = 10

input_data = pd.DataFrame({
    'week_of_year': [week_of_year],
    'day_of_week': [day_of_week], 
    'hour': [hour]
})

models = [model_k1, model_k2, model_k3]

wts = predict_cluster_weights(input_data, models)

print(f"Inputs\nWeek of year : {week_of_year}\nDay of week: {day_of_week}\nHour: {hour}")
for i in range(0, 3):
    print(f"Cluster {i+1} predicted weight: {wts[i]:.3f} , {wts[i]*100:.3f}%")

Inputs
Week of year : 51
Day of week: 3
Hour: 10
Cluster 1 predicted weight: 0.333 , 33.333%
Cluster 2 predicted weight: 0.333 , 33.333%
Cluster 3 predicted weight: 0.333 , 33.333%


Class 5

In [21]:
import pandas as pd
#filtering by class
class_num = 5
sample_ratio = .01
cols = [f"wt{i}" for i in range(1, 6)] #Max number of axles for this class is 5, determined in kmeans notebook

df_spark = df5.filter(f"class = {class_num}")

iqr_factor = 1.6
df_spark = remove_outliers_iqr2(df_spark, iqr_factor)[0]

Upper bound: 44052.0, Lower bound: -9162.0


In [22]:
max_iter = 20
num_clusters = 4
k3_model, df_spark2 = fit_gmm_spark(df_spark,cols, num_clusters, max_iter)

In [23]:
# Get the weights
weights = k3_model.weights

# Print them nicely
print("GMM Component Weights:")
for i, weight in enumerate(weights):
    print(f"Component {i}: {weight:.4f} ({weight * 100:.2f}%)")

GMM Component Weights:
Component 0: 0.2500 (25.00%)
Component 1: 0.2500 (25.00%)
Component 2: 0.2500 (25.00%)
Component 3: 0.2500 (25.00%)


In [61]:
df_spark3 = format_dt_inputs(df_spark2)

In [62]:
#Sample for parameter tuning
df_sample = df_spark3.sample(
    withReplacement=False,  
    fraction=sample_ratio,                          
).toPandas()

In [63]:
from sklearn.model_selection import train_test_split

input_cols = ["week_of_year", "day_of_week", "hour"]
output_cols = ["cluster1", "cluster2", "cluster3"]

df_sample[input_cols] = df_sample[input_cols].astype('category')
X_train, X_test, y_train, y_test = train_test_split(df_sample[input_cols], df_sample[output_cols], test_size=0.3)

print(f"Training set size: {X_train.shape[0]}")
print(f"Test set size: {X_test.shape[0]}")


Training set size: 25453
Test set size: 10909


In [64]:
import numpy as np
min_sample_leaf_array = ((np.arange(0.01, .11, .01))*X_train.shape[0]).astype("int")
min_sample_leaf_array = np.unique(min_sample_leaf_array)
max_depth_array = np.arange(10, 101, 10)

param_grid = {
    'min_samples_leaf' : min_sample_leaf_array,
    'max_depth' : max_depth_array
}

In [65]:
from sklearn.tree import DecisionTreeRegressor
from sklearn import tree
import matplotlib.pyplot as plt

dt_model = DecisionTreeRegressor()
input_cols = ["week_of_year", "day_of_week", "hour"]
output = "cluster1"

grid_search = dt_grid_search(X_train, y_train[output], X_test, y_test[output], dt_model, param_grid)

best_model = grid_search.best_estimator_
model_k1 = best_model

Fitting 5 folds for each of 100 candidates, totalling 500 fits
REGRESSION GRID SEARCH RESULTS
Best parameters: {'max_depth': np.int64(10), 'min_samples_leaf': np.int64(2545)}
Best CV Score (-MSE): -3.426746040194643e-19
Test MSE: 3.4400782197976514e-19


In [66]:
from sklearn.tree import export_graphviz
import graphviz

dot_data = export_graphviz(best_model, 
                           out_file=None,
                           feature_names=input_cols,
                           filled=True,
                           rounded=True,
                           special_characters=True,
                           precision=pc)

graph = graphviz.Source(dot_data)
graph.render(f"./decision_tree_vis/class_{class_num}/cluster1", format='pdf', cleanup=True)  # Creates PDF
# graph  # Display in notebook

'decision_tree_vis\\class_5\\cluster1.pdf'

In [67]:
from sklearn.tree import DecisionTreeRegressor
from sklearn import tree
import matplotlib.pyplot as plt

dt_model = DecisionTreeRegressor()
input_cols = ["week_of_year", "day_of_week", "hour"]
output = "cluster2"

grid_search = dt_grid_search(X_train, y_train[output], X_test, y_test[output], dt_model, param_grid)

best_model = grid_search.best_estimator_
model_k2 = best_model

Fitting 5 folds for each of 100 candidates, totalling 500 fits
REGRESSION GRID SEARCH RESULTS
Best parameters: {'max_depth': np.int64(10), 'min_samples_leaf': np.int64(254)}
Best CV Score (-MSE): -2.517908858835405e-18
Test MSE: 2.529567311075501e-18


In [68]:
from sklearn.tree import export_graphviz
import graphviz

dot_data = export_graphviz(best_model, 
                           out_file=None,
                           feature_names=input_cols,
                           filled=True,
                           rounded=True,
                           special_characters=True,
                           precision=pc)

graph = graphviz.Source(dot_data)
graph.render(f"./decision_tree_vis/class_{class_num}/cluster2", format='pdf', cleanup=True)  # Creates PDF
# graph  # Display in notebook

'decision_tree_vis\\class_5\\cluster2.pdf'

In [69]:
from sklearn.tree import DecisionTreeRegressor
from sklearn import tree
import matplotlib.pyplot as plt

dt_model = DecisionTreeRegressor()
input_cols = ["week_of_year", "day_of_week", "hour"]
output = "cluster3"

grid_search = dt_grid_search(X_train, y_train[output], X_test, y_test[output], dt_model, param_grid)

best_model = grid_search.best_estimator_
model_k3 = best_model

Fitting 5 folds for each of 100 candidates, totalling 500 fits
REGRESSION GRID SEARCH RESULTS
Best parameters: {'max_depth': np.int64(90), 'min_samples_leaf': np.int64(763)}
Best CV Score (-MSE): -4.717564816451159e-18
Test MSE: 4.733417722005018e-18


In [70]:
from sklearn.tree import export_graphviz
import graphviz

dot_data = export_graphviz(best_model, 
                           out_file=None,
                           feature_names=input_cols,
                           filled=True,
                           rounded=True,
                           special_characters=True,
                           precision=pc)

graph = graphviz.Source(dot_data)
graph.render(f"./decision_tree_vis/class_{class_num}/cluster3", format='pdf', cleanup=True)  # Creates PDF
# graph  # Display in notebook

'decision_tree_vis\\class_5\\cluster3.pdf'

In [71]:
#No error handling implemented yet care with args passed
week_of_year = 51
day_of_week = 3
hour = 10

input_data = pd.DataFrame({
    'week_of_year': [week_of_year],
    'day_of_week': [day_of_week], 
    'hour': [hour]
})

models = [model_k1, model_k2, model_k3]

wts = predict_cluster_weights(input_data, models)

print(f"Inputs\nWeek of year : {week_of_year}\nDay of week: {day_of_week}\nHour: {hour}")
for i in range(0, 3):
    print(f"Cluster {i+1} predicted weight: {wts[i]:.3f} , {wts[i]*100:.3f}%")

Inputs
Week of year : 51
Day of week: 3
Hour: 10
Cluster 1 predicted weight: 0.333 , 33.333%
Cluster 2 predicted weight: 0.333 , 33.333%
Cluster 3 predicted weight: 0.333 , 33.333%


Class 6

In [72]:
import pandas as pd
#filtering by class
class_num = 6
sample_ratio = .01
cols = [f"wt{i}" for i in range(1, 6)] #Max number of axles for this class is 5, determined in kmeans notebook

df_spark = df5.filter(f"class = {class_num}")

iqr_factor = 1.6
df_spark = remove_outliers_iqr2(df_spark, iqr_factor)[0]

Upper bound: 74704.0, Lower bound: -2534.0


In [73]:
max_iter = 20
num_clusters = 3
k3_model, df_spark2 = fit_gmm_spark(df_spark,cols, num_clusters, max_iter)

In [74]:
# Get the weights
weights = k3_model.weights

# Print them nicely
print("GMM Component Weights:")
for i, weight in enumerate(weights):
    print(f"Component {i}: {weight:.4f} ({weight * 100:.2f}%)")

GMM Component Weights:
Component 0: 0.3333 (33.33%)
Component 1: 0.3333 (33.33%)
Component 2: 0.3333 (33.33%)


In [75]:
df_spark3 = format_dt_inputs(df_spark2)

In [76]:
#Sample for parameter tuning
df_sample = df_spark3.sample(
    withReplacement=False,  
    fraction=sample_ratio,                          
).toPandas()

In [77]:
from sklearn.model_selection import train_test_split

input_cols = ["week_of_year", "day_of_week", "hour"]
output_cols = ["cluster1", "cluster2", "cluster3"]

df_sample[input_cols] = df_sample[input_cols].astype('category')
X_train, X_test, y_train, y_test = train_test_split(df_sample[input_cols], df_sample[output_cols], test_size=0.3)

print(f"Training set size: {X_train.shape[0]}")
print(f"Test set size: {X_test.shape[0]}")


Training set size: 7464
Test set size: 3199


In [78]:
import numpy as np
min_sample_leaf_array = ((np.arange(0.01, .11, .01))*X_train.shape[0]).astype("int")
min_sample_leaf_array = np.unique(min_sample_leaf_array)
max_depth_array = np.arange(10, 101, 10)

param_grid = {
    'min_samples_leaf' : min_sample_leaf_array,
    'max_depth' : max_depth_array
}

In [79]:
from sklearn.tree import DecisionTreeRegressor
from sklearn import tree
import matplotlib.pyplot as plt

dt_model = DecisionTreeRegressor()
input_cols = ["week_of_year", "day_of_week", "hour"]
output = "cluster1"

grid_search = dt_grid_search(X_train, y_train[output], X_test, y_test[output], dt_model, param_grid)

best_model = grid_search.best_estimator_
model_k1 = best_model

Fitting 5 folds for each of 100 candidates, totalling 500 fits
REGRESSION GRID SEARCH RESULTS
Best parameters: {'max_depth': np.int64(20), 'min_samples_leaf': np.int64(373)}
Best CV Score (-MSE): -1.942052885491186e-20
Test MSE: 1.9248919423305316e-20


In [80]:
from sklearn.tree import export_graphviz
import graphviz

dot_data = export_graphviz(best_model, 
                           out_file=None,
                           feature_names=input_cols,
                           filled=True,
                           rounded=True,
                           special_characters=True,
                           precision=pc)

graph = graphviz.Source(dot_data)
graph.render(f"./decision_tree_vis/class_{class_num}/cluster1", format='pdf', cleanup=True)  # Creates PDF
# graph  # Display in notebook

'decision_tree_vis\\class_6\\cluster1.pdf'

In [81]:
from sklearn.tree import DecisionTreeRegressor
from sklearn import tree
import matplotlib.pyplot as plt

dt_model = DecisionTreeRegressor()
input_cols = ["week_of_year", "day_of_week", "hour"]
output = "cluster2"

grid_search = dt_grid_search(X_train, y_train[output], X_test, y_test[output], dt_model, param_grid)

best_model = grid_search.best_estimator_
model_k2 = best_model

Fitting 5 folds for each of 100 candidates, totalling 500 fits
REGRESSION GRID SEARCH RESULTS
Best parameters: {'max_depth': np.int64(30), 'min_samples_leaf': np.int64(373)}
Best CV Score (-MSE): -1.9960551582773245e-19
Test MSE: 1.9754541691493492e-19


In [82]:
from sklearn.tree import export_graphviz
import graphviz

dot_data = export_graphviz(best_model, 
                           out_file=None,
                           feature_names=input_cols,
                           filled=True,
                           rounded=True,
                           special_characters=True,
                           precision=pc)

graph = graphviz.Source(dot_data)
graph.render(f"./decision_tree_vis/class_{class_num}/cluster2", format='pdf', cleanup=True)  # Creates PDF
# graph  # Display in notebook

'decision_tree_vis\\class_6\\cluster2.pdf'

In [83]:
from sklearn.tree import DecisionTreeRegressor
from sklearn import tree
import matplotlib.pyplot as plt

dt_model = DecisionTreeRegressor()
input_cols = ["week_of_year", "day_of_week", "hour"]
output = "cluster3"

grid_search = dt_grid_search(X_train, y_train[output], X_test, y_test[output], dt_model, param_grid)

best_model = grid_search.best_estimator_
model_k3 = best_model

Fitting 5 folds for each of 100 candidates, totalling 500 fits
REGRESSION GRID SEARCH RESULTS
Best parameters: {'max_depth': np.int64(10), 'min_samples_leaf': np.int64(74)}
Best CV Score (-MSE): -9.468528230829168e-20
Test MSE: 9.353843216781307e-20


In [84]:
from sklearn.tree import export_graphviz
import graphviz

dot_data = export_graphviz(best_model, 
                           out_file=None,
                           feature_names=input_cols,
                           filled=True,
                           rounded=True,
                           special_characters=True,
                           precision=pc)

graph = graphviz.Source(dot_data)
graph.render(f"./decision_tree_vis/class_{class_num}/cluster3", format='pdf', cleanup=True)  # Creates PDF
# graph  # Display in notebook

'decision_tree_vis\\class_6\\cluster3.pdf'

In [85]:
#No error handling implemented yet care with args passed
week_of_year = 51
day_of_week = 3
hour = 10

input_data = pd.DataFrame({
    'week_of_year': [week_of_year],
    'day_of_week': [day_of_week], 
    'hour': [hour]
})

models = [model_k1, model_k2, model_k3]

wts = predict_cluster_weights(input_data, models)

print(f"Inputs\nWeek of year : {week_of_year}\nDay of week: {day_of_week}\nHour: {hour}")
for i in range(0, 3):
    print(f"Cluster {i+1} predicted weight: {wts[i]:.3f} , {wts[i]*100:.3f}%")

Inputs
Week of year : 51
Day of week: 3
Hour: 10
Cluster 1 predicted weight: 0.333 , 33.333%
Cluster 2 predicted weight: 0.333 , 33.333%
Cluster 3 predicted weight: 0.333 , 33.333%


Class 7

In [86]:
import pandas as pd
#filtering by class
class_num = 7
sample_ratio = 1.0
cols = [f"wt{i}" for i in range(1, 8)] #Max number of axles for this class is 5, determined in kmeans notebook

df_spark = df5.filter(f"class = {class_num}")

iqr_factor = 1.5
df_spark = remove_outliers_iqr2(df_spark, iqr_factor)[0]

Upper bound: 115145.0, Lower bound: -1535.0


In [87]:
max_iter = 20
num_clusters = 3
k3_model, df_spark2 = fit_gmm_spark(df_spark,cols, num_clusters, max_iter)

In [88]:
# Get the weights
weights = k3_model.weights

# Print them nicely
print("GMM Component Weights:")
for i, weight in enumerate(weights):
    print(f"Component {i}: {weight:.4f} ({weight * 100:.2f}%)")

GMM Component Weights:
Component 0: 0.3333 (33.33%)
Component 1: 0.3333 (33.33%)
Component 2: 0.3333 (33.33%)


In [89]:
df_spark3 = format_dt_inputs(df_spark2)

In [90]:
#Sample for parameter tuning
df_sample = df_spark3.sample(
    withReplacement=False,  
    fraction=sample_ratio,                          
).toPandas()

In [91]:
from sklearn.model_selection import train_test_split

input_cols = ["week_of_year", "day_of_week", "hour"]
output_cols = ["cluster1", "cluster2", "cluster3"]

df_sample[input_cols] = df_sample[input_cols].astype('category')
X_train, X_test, y_train, y_test = train_test_split(df_sample[input_cols], df_sample[output_cols], test_size=0.3)

print(f"Training set size: {X_train.shape[0]}")
print(f"Test set size: {X_test.shape[0]}")


Training set size: 68463
Test set size: 29342


In [92]:
import numpy as np
min_sample_leaf_array = ((np.arange(0.01, .11, .01))*X_train.shape[0]).astype("int")
min_sample_leaf_array = np.unique(min_sample_leaf_array)
max_depth_array = np.arange(10, 101, 10)

param_grid = {
    'min_samples_leaf' : min_sample_leaf_array,
    'max_depth' : max_depth_array
}

In [93]:
from sklearn.tree import DecisionTreeRegressor
from sklearn import tree
import matplotlib.pyplot as plt

dt_model = DecisionTreeRegressor()
input_cols = ["week_of_year", "day_of_week", "hour"]
output = "cluster1"

grid_search = dt_grid_search(X_train, y_train[output], X_test, y_test[output], dt_model, param_grid)

best_model = grid_search.best_estimator_
model_k1 = best_model

Fitting 5 folds for each of 100 candidates, totalling 500 fits
REGRESSION GRID SEARCH RESULTS
Best parameters: {'max_depth': np.int64(90), 'min_samples_leaf': np.int64(4792)}
Best CV Score (-MSE): -2.6735728154936497e-27
Test MSE: 2.054161175084971e-26


In [94]:
from sklearn.tree import export_graphviz
import graphviz

dot_data = export_graphviz(best_model, 
                           out_file=None,
                           feature_names=input_cols,
                           filled=True,
                           rounded=True,
                           special_characters=True,
                           precision=pc)

graph = graphviz.Source(dot_data)
graph.render(f"./decision_tree_vis/class_{class_num}/cluster1", format='pdf', cleanup=True)  # Creates PDF
# graph  # Display in notebook

'decision_tree_vis\\class_7\\cluster1.pdf'

In [95]:
from sklearn.tree import DecisionTreeRegressor
from sklearn import tree
import matplotlib.pyplot as plt

dt_model = DecisionTreeRegressor()
input_cols = ["week_of_year", "day_of_week", "hour"]
output = "cluster2"

grid_search = dt_grid_search(X_train, y_train[output], X_test, y_test[output], dt_model, param_grid)

best_model = grid_search.best_estimator_
model_k2 = best_model

Fitting 5 folds for each of 100 candidates, totalling 500 fits
REGRESSION GRID SEARCH RESULTS
Best parameters: {'max_depth': np.int64(20), 'min_samples_leaf': np.int64(6846)}
Best CV Score (-MSE): -2.706743093742398e-27
Test MSE: 2.054161175084971e-26


In [96]:
from sklearn.tree import export_graphviz
import graphviz

dot_data = export_graphviz(best_model, 
                           out_file=None,
                           feature_names=input_cols,
                           filled=True,
                           rounded=True,
                           special_characters=True,
                           precision=pc)

graph = graphviz.Source(dot_data)
graph.render(f"./decision_tree_vis/class_{class_num}/cluster2", format='pdf', cleanup=True)  # Creates PDF
# graph  # Display in notebook

'decision_tree_vis\\class_7\\cluster2.pdf'

In [97]:
from sklearn.tree import DecisionTreeRegressor
from sklearn import tree
import matplotlib.pyplot as plt

dt_model = DecisionTreeRegressor()
input_cols = ["week_of_year", "day_of_week", "hour"]
output = "cluster3"

grid_search = dt_grid_search(X_train, y_train[output], X_test, y_test[output], dt_model, param_grid)

best_model = grid_search.best_estimator_
model_k3 = best_model

Fitting 5 folds for each of 100 candidates, totalling 500 fits
REGRESSION GRID SEARCH RESULTS
Best parameters: {'max_depth': np.int64(100), 'min_samples_leaf': np.int64(6161)}
Best CV Score (-MSE): -3.3434137777884294e-27
Test MSE: 2.276650454955716e-27


In [98]:
from sklearn.tree import export_graphviz
import graphviz

dot_data = export_graphviz(best_model, 
                           out_file=None,
                           feature_names=input_cols,
                           filled=True,
                           rounded=True,
                           special_characters=True,
                           precision=pc)

graph = graphviz.Source(dot_data)
graph.render(f"./decision_tree_vis/class_{class_num}/cluster3", format='pdf', cleanup=True)  # Creates PDF
# graph  # Display in notebook

'decision_tree_vis\\class_7\\cluster3.pdf'

In [99]:
#No error handling implemented yet care with args passed
week_of_year = 51
day_of_week = 3
hour = 10

input_data = pd.DataFrame({
    'week_of_year': [week_of_year],
    'day_of_week': [day_of_week], 
    'hour': [hour]
})

models = [model_k1, model_k2, model_k3]

wts = predict_cluster_weights(input_data, models)

print(f"Inputs\nWeek of year : {week_of_year}\nDay of week: {day_of_week}\nHour: {hour}")
for i in range(0, 3):
    print(f"Cluster {i+1} predicted weight: {wts[i]:.3f} , {wts[i]*100:.3f}%")

Inputs
Week of year : 51
Day of week: 3
Hour: 10
Cluster 1 predicted weight: 0.333 , 33.333%
Cluster 2 predicted weight: 0.333 , 33.333%
Cluster 3 predicted weight: 0.333 , 33.333%


Class 8

In [100]:
import pandas as pd
#filtering by class
class_num = 8
sample_ratio = 1.0
cols = [f"wt{i}" for i in range(1, 5)] #Max number of axles for this class is 5, determined in kmeans notebook

df_spark = df5.filter(f"class = {class_num}")

iqr_factor = 1.5
df_spark = remove_outliers_iqr2(df_spark, iqr_factor)[0]

Upper bound: 76745.0, Lower bound: 5985.0


In [101]:
max_iter = 20
num_clusters = 3
k3_model, df_spark2 = fit_gmm_spark(df_spark,cols, num_clusters, max_iter)

In [102]:
# Get the weights
weights = k3_model.weights

# Print them nicely
print("GMM Component Weights:")
for i, weight in enumerate(weights):
    print(f"Component {i}: {weight:.4f} ({weight * 100:.2f}%)")

GMM Component Weights:
Component 0: 0.3333 (33.33%)
Component 1: 0.3333 (33.33%)
Component 2: 0.3333 (33.33%)


In [103]:
df_spark3 = format_dt_inputs(df_spark2)

In [104]:
#Sample for parameter tuning
df_sample = df_spark3.sample(
    withReplacement=False,  
    fraction=sample_ratio,                          
).toPandas()

In [105]:
from sklearn.model_selection import train_test_split

input_cols = ["week_of_year", "day_of_week", "hour"]
output_cols = ["cluster1", "cluster2", "cluster3"]

df_sample[input_cols] = df_sample[input_cols].astype('category')
X_train, X_test, y_train, y_test = train_test_split(df_sample[input_cols], df_sample[output_cols], test_size=0.3)

print(f"Training set size: {X_train.shape[0]}")
print(f"Test set size: {X_test.shape[0]}")


Training set size: 383754
Test set size: 164467


In [106]:
import numpy as np
min_sample_leaf_array = ((np.arange(0.01, .11, .01))*X_train.shape[0]).astype("int")
min_sample_leaf_array = np.unique(min_sample_leaf_array)
max_depth_array = np.arange(10, 101, 10)

param_grid = {
    'min_samples_leaf' : min_sample_leaf_array,
    'max_depth' : max_depth_array
}

In [107]:
from sklearn.tree import DecisionTreeRegressor
from sklearn import tree
import matplotlib.pyplot as plt

dt_model = DecisionTreeRegressor()
input_cols = ["week_of_year", "day_of_week", "hour"]
output = "cluster1"

grid_search = dt_grid_search(X_train, y_train[output], X_test, y_test[output], dt_model, param_grid)

best_model = grid_search.best_estimator_
model_k1 = best_model

Fitting 5 folds for each of 100 candidates, totalling 500 fits
REGRESSION GRID SEARCH RESULTS
Best parameters: {'max_depth': np.int64(30), 'min_samples_leaf': np.int64(19187)}
Best CV Score (-MSE): -5.057862662991612e-15
Test MSE: 5.0564372304222886e-15


In [108]:
from sklearn.tree import export_graphviz
import graphviz

dot_data = export_graphviz(best_model, 
                           out_file=None,
                           feature_names=input_cols,
                           filled=True,
                           rounded=True,
                           special_characters=True,
                           precision=pc)

graph = graphviz.Source(dot_data)
graph.render(f"./decision_tree_vis/class_{class_num}/cluster1", format='pdf', cleanup=True)  # Creates PDF
# graph  # Display in notebook

'decision_tree_vis\\class_8\\cluster1.pdf'

In [109]:
from sklearn.tree import DecisionTreeRegressor
from sklearn import tree
import matplotlib.pyplot as plt

dt_model = DecisionTreeRegressor()
input_cols = ["week_of_year", "day_of_week", "hour"]
output = "cluster2"

grid_search = dt_grid_search(X_train, y_train[output], X_test, y_test[output], dt_model, param_grid)

best_model = grid_search.best_estimator_
model_k2 = best_model

Fitting 5 folds for each of 100 candidates, totalling 500 fits
REGRESSION GRID SEARCH RESULTS
Best parameters: {'max_depth': np.int64(70), 'min_samples_leaf': np.int64(3837)}
Best CV Score (-MSE): -8.534926896467956e-15
Test MSE: 8.509784029177309e-15


In [110]:
from sklearn.tree import export_graphviz
import graphviz

dot_data = export_graphviz(best_model, 
                           out_file=None,
                           feature_names=input_cols,
                           filled=True,
                           rounded=True,
                           special_characters=True,
                           precision=pc)

graph = graphviz.Source(dot_data)
graph.render(f"./decision_tree_vis/class_{class_num}/cluster2", format='pdf', cleanup=True)  # Creates PDF
# graph  # Display in notebook

'decision_tree_vis\\class_8\\cluster2.pdf'

In [111]:
from sklearn.tree import DecisionTreeRegressor
from sklearn import tree
import matplotlib.pyplot as plt

dt_model = DecisionTreeRegressor()
input_cols = ["week_of_year", "day_of_week", "hour"]
output = "cluster3"

grid_search = dt_grid_search(X_train, y_train[output], X_test, y_test[output], dt_model, param_grid)

best_model = grid_search.best_estimator_
model_k3 = best_model

Fitting 5 folds for each of 100 candidates, totalling 500 fits
REGRESSION GRID SEARCH RESULTS
Best parameters: {'max_depth': np.int64(50), 'min_samples_leaf': np.int64(34537)}
Best CV Score (-MSE): -2.704776003126054e-14
Test MSE: 2.7284808954677998e-14


In [112]:
from sklearn.tree import export_graphviz
import graphviz

dot_data = export_graphviz(best_model, 
                           out_file=None,
                           feature_names=input_cols,
                           filled=True,
                           rounded=True,
                           special_characters=True,
                           precision=pc)

graph = graphviz.Source(dot_data)
graph.render(f"./decision_tree_vis/class_{class_num}/cluster3", format='pdf', cleanup=True)  # Creates PDF
# graph  # Display in notebook

'decision_tree_vis\\class_8\\cluster3.pdf'

In [113]:
#No error handling implemented yet care with args passed
week_of_year = 51
day_of_week = 3
hour = 10

input_data = pd.DataFrame({
    'week_of_year': [week_of_year],
    'day_of_week': [day_of_week], 
    'hour': [hour]
})

models = [model_k1, model_k2, model_k3]

wts = predict_cluster_weights(input_data, models)

print(f"Inputs\nWeek of year : {week_of_year}\nDay of week: {day_of_week}\nHour: {hour}")
for i in range(0, 3):
    print(f"Cluster {i+1} predicted weight: {wts[i]:.3f} , {wts[i]*100:.3f}%")

Inputs
Week of year : 51
Day of week: 3
Hour: 10
Cluster 1 predicted weight: 0.333 , 33.333%
Cluster 2 predicted weight: 0.333 , 33.333%
Cluster 3 predicted weight: 0.333 , 33.333%


Class 9

In [114]:
import pandas as pd
#filtering by class
class_num = 9
sample_ratio = .01
cols = [f"wt{i}" for i in range(1, 6)] #Max number of axles for this class is 5, determined in kmeans notebook

df_spark = df5.filter(f"class = {class_num}")

iqr_factor = 1.5
df_spark = remove_outliers_iqr2(df_spark, iqr_factor)[0]

Upper bound: 121400.0, Lower bound: -1800.0


In [115]:
max_iter = 20
num_clusters = 3
k3_model, df_spark2 = fit_gmm_spark(df_spark,cols, num_clusters, max_iter)

In [116]:
# Get the weights
weights = k3_model.weights

# Print them nicely
print("GMM Component Weights:")
for i, weight in enumerate(weights):
    print(f"Component {i}: {weight:.4f} ({weight * 100:.2f}%)")

GMM Component Weights:
Component 0: 0.3333 (33.33%)
Component 1: 0.3333 (33.33%)
Component 2: 0.3333 (33.33%)


In [117]:
df_spark3 = format_dt_inputs(df_spark2)

In [118]:
#Sample for parameter tuning
df_sample = df_spark3.sample(
    withReplacement=False,  
    fraction=sample_ratio,                          
).toPandas()

In [119]:
from sklearn.model_selection import train_test_split

input_cols = ["week_of_year", "day_of_week", "hour"]
output_cols = ["cluster1", "cluster2", "cluster3"]

df_sample[input_cols] = df_sample[input_cols].astype('category')
X_train, X_test, y_train, y_test = train_test_split(df_sample[input_cols], df_sample[output_cols], test_size=0.3)

print(f"Training set size: {X_train.shape[0]}")
print(f"Test set size: {X_test.shape[0]}")


Training set size: 16130
Test set size: 6913


In [120]:
import numpy as np
min_sample_leaf_array = ((np.arange(0.01, .11, .01))*X_train.shape[0]).astype("int")
min_sample_leaf_array = np.unique(min_sample_leaf_array)
max_depth_array = np.arange(10, 101, 10)

param_grid = {
    'min_samples_leaf' : min_sample_leaf_array,
    'max_depth' : max_depth_array
}

In [121]:
from sklearn.tree import DecisionTreeRegressor
from sklearn import tree
import matplotlib.pyplot as plt

dt_model = DecisionTreeRegressor()
input_cols = ["week_of_year", "day_of_week", "hour"]
output = "cluster1"

grid_search = dt_grid_search(X_train, y_train[output], X_test, y_test[output], dt_model, param_grid)

best_model = grid_search.best_estimator_
model_k1 = best_model

Fitting 5 folds for each of 100 candidates, totalling 500 fits
REGRESSION GRID SEARCH RESULTS
Best parameters: {'max_depth': np.int64(10), 'min_samples_leaf': np.int64(161)}
Best CV Score (-MSE): -1.861899640219884e-27
Test MSE: 6.82767199421441e-28


In [122]:
from sklearn.tree import export_graphviz
import graphviz

dot_data = export_graphviz(best_model, 
                           out_file=None,
                           feature_names=input_cols,
                           filled=True,
                           rounded=True,
                           special_characters=True,
                           precision=pc)

graph = graphviz.Source(dot_data)
graph.render(f"./decision_tree_vis/class_{class_num}/cluster1", format='pdf', cleanup=True)  # Creates PDF
# graph  # Display in notebook

'decision_tree_vis\\class_9\\cluster1.pdf'

In [123]:
from sklearn.tree import DecisionTreeRegressor
from sklearn import tree
import matplotlib.pyplot as plt

dt_model = DecisionTreeRegressor()
input_cols = ["week_of_year", "day_of_week", "hour"]
output = "cluster2"

grid_search = dt_grid_search(X_train, y_train[output], X_test, y_test[output], dt_model, param_grid)

best_model = grid_search.best_estimator_
model_k2 = best_model

Fitting 5 folds for each of 100 candidates, totalling 500 fits
REGRESSION GRID SEARCH RESULTS
Best parameters: {'max_depth': np.int64(10), 'min_samples_leaf': np.int64(161)}
Best CV Score (-MSE): -1.8580515195724542e-27
Test MSE: 6.851785432786578e-28


In [124]:
from sklearn.tree import export_graphviz
import graphviz

dot_data = export_graphviz(best_model, 
                           out_file=None,
                           feature_names=input_cols,
                           filled=True,
                           rounded=True,
                           special_characters=True,
                           precision=pc)

graph = graphviz.Source(dot_data)
graph.render(f"./decision_tree_vis/class_{class_num}/cluster2", format='pdf', cleanup=True)  # Creates PDF
# graph  # Display in notebook

'decision_tree_vis\\class_9\\cluster2.pdf'

In [125]:
from sklearn.tree import DecisionTreeRegressor
from sklearn import tree
import matplotlib.pyplot as plt

dt_model = DecisionTreeRegressor()
input_cols = ["week_of_year", "day_of_week", "hour"]
output = "cluster3"

grid_search = dt_grid_search(X_train, y_train[output], X_test, y_test[output], dt_model, param_grid)

best_model = grid_search.best_estimator_
model_k3 = best_model

Fitting 5 folds for each of 100 candidates, totalling 500 fits
REGRESSION GRID SEARCH RESULTS
Best parameters: {'max_depth': np.int64(10), 'min_samples_leaf': np.int64(161)}
Best CV Score (-MSE): -1.8855600240513408e-27
Test MSE: 6.684104407091229e-28


In [126]:
from sklearn.tree import export_graphviz
import graphviz

dot_data = export_graphviz(best_model, 
                           out_file=None,
                           feature_names=input_cols,
                           filled=True,
                           rounded=True,
                           special_characters=True,
                           precision=pc)

graph = graphviz.Source(dot_data)
graph.render(f"./decision_tree_vis/class_{class_num}/cluster3", format='pdf', cleanup=True)  # Creates PDF
# graph  # Display in notebook

'decision_tree_vis\\class_9\\cluster3.pdf'

In [127]:
#No error handling implemented yet care with args passed
week_of_year = 51
day_of_week = 3
hour = 10

input_data = pd.DataFrame({
    'week_of_year': [week_of_year],
    'day_of_week': [day_of_week], 
    'hour': [hour]
})

models = [model_k1, model_k2, model_k3]

wts = predict_cluster_weights(input_data, models)

print(f"Inputs\nWeek of year : {week_of_year}\nDay of week: {day_of_week}\nHour: {hour}")
for i in range(0, 3):
    print(f"Cluster {i+1} predicted weight: {wts[i]:.3f} , {wts[i]*100:.3f}%")

Inputs
Week of year : 51
Day of week: 3
Hour: 10
Cluster 1 predicted weight: 0.333 , 33.333%
Cluster 2 predicted weight: 0.333 , 33.333%
Cluster 3 predicted weight: 0.333 , 33.333%


Class 10

In [128]:
import pandas as pd
#filtering by class
class_num = 10
sample_ratio = 1.0
cols = [f"wt{i}" for i in range(1, 18)] #Max number of axles for this class is 5, determined in kmeans notebook

df_spark = df5.filter(f"class = {class_num}")

iqr_factor = 1.5
df_spark = remove_outliers_iqr2(df_spark, iqr_factor)[0]

Upper bound: 175600.0, Lower bound: 3280.0


In [129]:
max_iter = 20
num_clusters = 3
k3_model, df_spark2 = fit_gmm_spark(df_spark,cols, num_clusters, max_iter)

In [130]:
# Get the weights
weights = k3_model.weights

# Print them nicely
print("GMM Component Weights:")
for i, weight in enumerate(weights):
    print(f"Component {i}: {weight:.4f} ({weight * 100:.2f}%)")

GMM Component Weights:
Component 0: 0.3333 (33.33%)
Component 1: 0.3333 (33.33%)
Component 2: 0.3333 (33.33%)


In [131]:
df_spark3 = format_dt_inputs(df_spark2)

In [132]:
#Sample for parameter tuning
df_sample = df_spark3.sample(
    withReplacement=False,  
    fraction=sample_ratio,                          
).toPandas()

In [133]:
from sklearn.model_selection import train_test_split

input_cols = ["week_of_year", "day_of_week", "hour"]
output_cols = ["cluster1", "cluster2", "cluster3"]

df_sample[input_cols] = df_sample[input_cols].astype('category')
X_train, X_test, y_train, y_test = train_test_split(df_sample[input_cols], df_sample[output_cols], test_size=0.3)

print(f"Training set size: {X_train.shape[0]}")
print(f"Test set size: {X_test.shape[0]}")


Training set size: 65145
Test set size: 27920


In [134]:
import numpy as np
min_sample_leaf_array = ((np.arange(0.01, .11, .01))*X_train.shape[0]).astype("int")
min_sample_leaf_array = np.unique(min_sample_leaf_array)
max_depth_array = np.arange(10, 101, 10)

param_grid = {
    'min_samples_leaf' : min_sample_leaf_array,
    'max_depth' : max_depth_array
}

In [135]:
from sklearn.tree import DecisionTreeRegressor
from sklearn import tree
import matplotlib.pyplot as plt

dt_model = DecisionTreeRegressor()
input_cols = ["week_of_year", "day_of_week", "hour"]
output = "cluster1"

grid_search = dt_grid_search(X_train, y_train[output], X_test, y_test[output], dt_model, param_grid)

best_model = grid_search.best_estimator_
model_k1 = best_model

Fitting 5 folds for each of 100 candidates, totalling 500 fits
REGRESSION GRID SEARCH RESULTS
Best parameters: {'max_depth': np.int64(100), 'min_samples_leaf': np.int64(6514)}
Best CV Score (-MSE): -3.1881429564555986e-27
Test MSE: 2.421781854943255e-27


In [136]:
from sklearn.tree import export_graphviz
import graphviz

dot_data = export_graphviz(best_model, 
                           out_file=None,
                           feature_names=input_cols,
                           filled=True,
                           rounded=True,
                           special_characters=True,
                           precision=pc)

graph = graphviz.Source(dot_data)
graph.render(f"./decision_tree_vis/class_{class_num}/cluster1", format='pdf', cleanup=True)  # Creates PDF
# graph  # Display in notebook

'decision_tree_vis\\class_10\\cluster1.pdf'

In [137]:
from sklearn.tree import DecisionTreeRegressor
from sklearn import tree
import matplotlib.pyplot as plt

dt_model = DecisionTreeRegressor()
input_cols = ["week_of_year", "day_of_week", "hour"]
output = "cluster2"

grid_search = dt_grid_search(X_train, y_train[output], X_test, y_test[output], dt_model, param_grid)

best_model = grid_search.best_estimator_
model_k2 = best_model

Fitting 5 folds for each of 100 candidates, totalling 500 fits
REGRESSION GRID SEARCH RESULTS
Best parameters: {'max_depth': np.int64(70), 'min_samples_leaf': np.int64(5211)}
Best CV Score (-MSE): -3.1171256084824882e-27
Test MSE: 8.348760363161536e-28


In [138]:
from sklearn.tree import export_graphviz
import graphviz

dot_data = export_graphviz(best_model, 
                           out_file=None,
                           feature_names=input_cols,
                           filled=True,
                           rounded=True,
                           special_characters=True,
                           precision=pc)

graph = graphviz.Source(dot_data)
graph.render(f"./decision_tree_vis/class_{class_num}/cluster2", format='pdf', cleanup=True)  # Creates PDF
# graph  # Display in notebook

'decision_tree_vis\\class_10\\cluster2.pdf'

In [139]:
from sklearn.tree import DecisionTreeRegressor
from sklearn import tree
import matplotlib.pyplot as plt

dt_model = DecisionTreeRegressor()
input_cols = ["week_of_year", "day_of_week", "hour"]
output = "cluster3"

grid_search = dt_grid_search(X_train, y_train[output], X_test, y_test[output], dt_model, param_grid)

best_model = grid_search.best_estimator_
model_k3 = best_model

Fitting 5 folds for each of 100 candidates, totalling 500 fits
REGRESSION GRID SEARCH RESULTS
Best parameters: {'max_depth': np.int64(100), 'min_samples_leaf': np.int64(5863)}
Best CV Score (-MSE): -3.267680519499619e-27
Test MSE: 1.8427726544729443e-26


In [140]:
from sklearn.tree import export_graphviz
import graphviz

dot_data = export_graphviz(best_model, 
                           out_file=None,
                           feature_names=input_cols,
                           filled=True,
                           rounded=True,
                           special_characters=True,
                           precision=pc)

graph = graphviz.Source(dot_data)
graph.render(f"./decision_tree_vis/class_{class_num}/cluster3", format='pdf', cleanup=True)  # Creates PDF
# graph  # Display in notebook

'decision_tree_vis\\class_10\\cluster3.pdf'

In [141]:
#No error handling implemented yet care with args passed
week_of_year = 51
day_of_week = 3
hour = 10

input_data = pd.DataFrame({
    'week_of_year': [week_of_year],
    'day_of_week': [day_of_week], 
    'hour': [hour]
})

models = [model_k1, model_k2, model_k3]

wts = predict_cluster_weights(input_data, models)

print(f"Inputs\nWeek of year : {week_of_year}\nDay of week: {day_of_week}\nHour: {hour}")
for i in range(0, 3):
    print(f"Cluster {i+1} predicted weight: {wts[i]:.3f} , {wts[i]*100:.3f}%")

Inputs
Week of year : 51
Day of week: 3
Hour: 10
Cluster 1 predicted weight: 0.333 , 33.333%
Cluster 2 predicted weight: 0.333 , 33.333%
Cluster 3 predicted weight: 0.333 , 33.333%


Class 11

In [142]:
import pandas as pd
#filtering by class
class_num = 11
sample_ratio = 1.0
cols = [f"wt{i}" for i in range(1, 6)] #Max number of axles for this class is 5, determined in kmeans notebook

df_spark = df5.filter(f"class = {class_num}")

iqr_factor = 1.7
df_spark = remove_outliers_iqr2(df_spark, iqr_factor)[0]

Upper bound: 109110.0, Lower bound: 20670.0


In [143]:
max_iter = 20
num_clusters = 3
k3_model, df_spark2 = fit_gmm_spark(df_spark,cols, num_clusters, max_iter)

In [144]:
# Get the weights
weights = k3_model.weights

# Print them nicely
print("GMM Component Weights:")
for i, weight in enumerate(weights):
    print(f"Component {i}: {weight:.4f} ({weight * 100:.2f}%)")

GMM Component Weights:
Component 0: 0.3333 (33.33%)
Component 1: 0.3333 (33.33%)
Component 2: 0.3333 (33.33%)


In [145]:
df_spark3 = format_dt_inputs(df_spark2)

In [146]:
#Sample for parameter tuning
df_sample = df_spark3.sample(
    withReplacement=False,  
    fraction=sample_ratio,                          
).toPandas()

In [147]:
from sklearn.model_selection import train_test_split

input_cols = ["week_of_year", "day_of_week", "hour"]
output_cols = ["cluster1", "cluster2", "cluster3"]

df_sample[input_cols] = df_sample[input_cols].astype('category')
X_train, X_test, y_train, y_test = train_test_split(df_sample[input_cols], df_sample[output_cols], test_size=0.3)

print(f"Training set size: {X_train.shape[0]}")
print(f"Test set size: {X_test.shape[0]}")


Training set size: 48510
Test set size: 20791


In [148]:
import numpy as np
min_sample_leaf_array = ((np.arange(0.01, .11, .01))*X_train.shape[0]).astype("int")
min_sample_leaf_array = np.unique(min_sample_leaf_array)
max_depth_array = np.arange(10, 101, 10)

param_grid = {
    'min_samples_leaf' : min_sample_leaf_array,
    'max_depth' : max_depth_array
}

In [149]:
from sklearn.tree import DecisionTreeRegressor
from sklearn import tree
import matplotlib.pyplot as plt

dt_model = DecisionTreeRegressor()
input_cols = ["week_of_year", "day_of_week", "hour"]
output = "cluster1"

grid_search = dt_grid_search(X_train, y_train[output], X_test, y_test[output], dt_model, param_grid)

best_model = grid_search.best_estimator_
model_k1 = best_model

Fitting 5 folds for each of 100 candidates, totalling 500 fits
REGRESSION GRID SEARCH RESULTS
Best parameters: {'max_depth': np.int64(10), 'min_samples_leaf': np.int64(485)}
Best CV Score (-MSE): -2.1133644136648178e-26
Test MSE: 5.64158054741026e-26


In [150]:
from sklearn.tree import export_graphviz
import graphviz

dot_data = export_graphviz(best_model, 
                           out_file=None,
                           feature_names=input_cols,
                           filled=True,
                           rounded=True,
                           special_characters=True,
                           precision=pc)

graph = graphviz.Source(dot_data)
graph.render(f"./decision_tree_vis/class_{class_num}/cluster1", format='pdf', cleanup=True)  # Creates PDF
# graph  # Display in notebook

'decision_tree_vis\\class_11\\cluster1.pdf'

In [151]:
from sklearn.tree import DecisionTreeRegressor
from sklearn import tree
import matplotlib.pyplot as plt

dt_model = DecisionTreeRegressor()
input_cols = ["week_of_year", "day_of_week", "hour"]
output = "cluster2"

grid_search = dt_grid_search(X_train, y_train[output], X_test, y_test[output], dt_model, param_grid)

best_model = grid_search.best_estimator_
model_k2 = best_model

Fitting 5 folds for each of 100 candidates, totalling 500 fits
REGRESSION GRID SEARCH RESULTS
Best parameters: {'max_depth': np.int64(10), 'min_samples_leaf': np.int64(485)}
Best CV Score (-MSE): -2.1201489508487972e-26
Test MSE: 5.652825513347115e-26


In [152]:
from sklearn.tree import export_graphviz
import graphviz

dot_data = export_graphviz(best_model, 
                           out_file=None,
                           feature_names=input_cols,
                           filled=True,
                           rounded=True,
                           special_characters=True,
                           precision=pc)

graph = graphviz.Source(dot_data)
graph.render(f"./decision_tree_vis/class_{class_num}/cluster2", format='pdf', cleanup=True)  # Creates PDF
# graph  # Display in notebook

'decision_tree_vis\\class_11\\cluster2.pdf'

In [153]:
from sklearn.tree import DecisionTreeRegressor
from sklearn import tree
import matplotlib.pyplot as plt

dt_model = DecisionTreeRegressor()
input_cols = ["week_of_year", "day_of_week", "hour"]
output = "cluster3"

grid_search = dt_grid_search(X_train, y_train[output], X_test, y_test[output], dt_model, param_grid)

best_model = grid_search.best_estimator_
model_k3 = best_model

Fitting 5 folds for each of 100 candidates, totalling 500 fits
REGRESSION GRID SEARCH RESULTS
Best parameters: {'max_depth': np.int64(10), 'min_samples_leaf': np.int64(485)}
Best CV Score (-MSE): -2.1204381181380833e-26
Test MSE: 5.653311611225067e-26


In [154]:
from sklearn.tree import export_graphviz
import graphviz

dot_data = export_graphviz(best_model, 
                           out_file=None,
                           feature_names=input_cols,
                           filled=True,
                           rounded=True,
                           special_characters=True,
                           precision=pc)

graph = graphviz.Source(dot_data)
graph.render(f"./decision_tree_vis/class_{class_num}/cluster3", format='pdf', cleanup=True)  # Creates PDF
# graph  # Display in notebook

'decision_tree_vis\\class_11\\cluster3.pdf'

In [155]:
#No error handling implemented yet care with args passed
week_of_year = 51
day_of_week = 3
hour = 10

input_data = pd.DataFrame({
    'week_of_year': [week_of_year],
    'day_of_week': [day_of_week], 
    'hour': [hour]
})

models = [model_k1, model_k2, model_k3]

wts = predict_cluster_weights(input_data, models)

print(f"Inputs\nWeek of year : {week_of_year}\nDay of week: {day_of_week}\nHour: {hour}")
for i in range(0, 3):
    print(f"Cluster {i+1} predicted weight: {wts[i]:.3f} , {wts[i]*100:.3f}%")

Inputs
Week of year : 51
Day of week: 3
Hour: 10
Cluster 1 predicted weight: 0.333 , 33.333%
Cluster 2 predicted weight: 0.333 , 33.333%
Cluster 3 predicted weight: 0.333 , 33.333%


Class 12

In [156]:
import pandas as pd
#filtering by class
class_num = 12
sample_ratio = 1.0
cols = [f"wt{i}" for i in range(1, 7)] #Max number of axles for this class is 5, determined in kmeans notebook

df_spark = df5.filter(f"class = {class_num}")

iqr_factor = 2.0
df_spark = remove_outliers_iqr2(df_spark, iqr_factor)[0]

Upper bound: 100380.0, Lower bound: 19980.0


In [157]:
max_iter = 20
num_clusters = 3
k3_model, df_spark2 = fit_gmm_spark(df_spark,cols, num_clusters, max_iter)

In [158]:
# Get the weights
weights = k3_model.weights

# Print them nicely
print("GMM Component Weights:")
for i, weight in enumerate(weights):
    print(f"Component {i}: {weight:.4f} ({weight * 100:.2f}%)")

GMM Component Weights:
Component 0: 0.3333 (33.33%)
Component 1: 0.3333 (33.33%)
Component 2: 0.3333 (33.33%)


In [159]:
df_spark3 = format_dt_inputs(df_spark2)

In [160]:
#Sample for parameter tuning
df_sample = df_spark3.sample(
    withReplacement=False,  
    fraction=sample_ratio,                          
).toPandas()

In [161]:
from sklearn.model_selection import train_test_split

input_cols = ["week_of_year", "day_of_week", "hour"]
output_cols = ["cluster1", "cluster2", "cluster3"]

df_sample[input_cols] = df_sample[input_cols].astype('category')
X_train, X_test, y_train, y_test = train_test_split(df_sample[input_cols], df_sample[output_cols], test_size=0.3)

print(f"Training set size: {X_train.shape[0]}")
print(f"Test set size: {X_test.shape[0]}")


Training set size: 22626
Test set size: 9697


In [162]:
import numpy as np
min_sample_leaf_array = ((np.arange(0.01, .11, .01))*X_train.shape[0]).astype("int")
min_sample_leaf_array = np.unique(min_sample_leaf_array)
max_depth_array = np.arange(10, 101, 10)

param_grid = {
    'min_samples_leaf' : min_sample_leaf_array,
    'max_depth' : max_depth_array
}

In [163]:
from sklearn.tree import DecisionTreeRegressor
from sklearn import tree
import matplotlib.pyplot as plt

dt_model = DecisionTreeRegressor()
input_cols = ["week_of_year", "day_of_week", "hour"]
output = "cluster1"

grid_search = dt_grid_search(X_train, y_train[output], X_test, y_test[output], dt_model, param_grid)

best_model = grid_search.best_estimator_
model_k1 = best_model

Fitting 5 folds for each of 100 candidates, totalling 500 fits
REGRESSION GRID SEARCH RESULTS
Best parameters: {'max_depth': np.int64(60), 'min_samples_leaf': np.int64(226)}
Best CV Score (-MSE): -3.0900936243488667e-28
Test MSE: 1.2205511941709181e-27


In [164]:
from sklearn.tree import export_graphviz
import graphviz

dot_data = export_graphviz(best_model, 
                           out_file=None,
                           feature_names=input_cols,
                           filled=True,
                           rounded=True,
                           special_characters=True,
                           precision=pc)

graph = graphviz.Source(dot_data)
graph.render(f"./decision_tree_vis/class_{class_num}/cluster1", format='pdf', cleanup=True)  # Creates PDF
# graph  # Display in notebook

'decision_tree_vis\\class_12\\cluster1.pdf'

In [165]:
from sklearn.tree import DecisionTreeRegressor
from sklearn import tree
import matplotlib.pyplot as plt

dt_model = DecisionTreeRegressor()
input_cols = ["week_of_year", "day_of_week", "hour"]
output = "cluster2"

grid_search = dt_grid_search(X_train, y_train[output], X_test, y_test[output], dt_model, param_grid)

best_model = grid_search.best_estimator_
model_k2 = best_model

Fitting 5 folds for each of 100 candidates, totalling 500 fits
REGRESSION GRID SEARCH RESULTS
Best parameters: {'max_depth': np.int64(30), 'min_samples_leaf': np.int64(226)}
Best CV Score (-MSE): -4.396181791930334e-28
Test MSE: 1.219722603458067e-27


In [166]:
from sklearn.tree import export_graphviz
import graphviz

dot_data = export_graphviz(best_model, 
                           out_file=None,
                           feature_names=input_cols,
                           filled=True,
                           rounded=True,
                           special_characters=True,
                           precision=pc)

graph = graphviz.Source(dot_data)
graph.render(f"./decision_tree_vis/class_{class_num}/cluster2", format='pdf', cleanup=True)  # Creates PDF
# graph  # Display in notebook

'decision_tree_vis\\class_12\\cluster2.pdf'

In [167]:
from sklearn.tree import DecisionTreeRegressor
from sklearn import tree
import matplotlib.pyplot as plt

dt_model = DecisionTreeRegressor()
input_cols = ["week_of_year", "day_of_week", "hour"]
output = "cluster3"

grid_search = dt_grid_search(X_train, y_train[output], X_test, y_test[output], dt_model, param_grid)

best_model = grid_search.best_estimator_
model_k3 = best_model

Fitting 5 folds for each of 100 candidates, totalling 500 fits
REGRESSION GRID SEARCH RESULTS
Best parameters: {'max_depth': np.int64(30), 'min_samples_leaf': np.int64(226)}
Best CV Score (-MSE): -4.260098938328036e-28
Test MSE: 1.0570615438086895e-28


In [168]:
from sklearn.tree import export_graphviz
import graphviz

dot_data = export_graphviz(best_model, 
                           out_file=None,
                           feature_names=input_cols,
                           filled=True,
                           rounded=True,
                           special_characters=True,
                           precision=pc)

graph = graphviz.Source(dot_data)
graph.render(f"./decision_tree_vis/class_{class_num}/cluster3", format='pdf', cleanup=True)  # Creates PDF
# graph  # Display in notebook

'decision_tree_vis\\class_12\\cluster3.pdf'

In [169]:
#No error handling implemented yet care with args passed
week_of_year = 51
day_of_week = 3
hour = 10

input_data = pd.DataFrame({
    'week_of_year': [week_of_year],
    'day_of_week': [day_of_week], 
    'hour': [hour]
})

models = [model_k1, model_k2, model_k3]

wts = predict_cluster_weights(input_data, models)

print(f"Inputs\nWeek of year : {week_of_year}\nDay of week: {day_of_week}\nHour: {hour}")
for i in range(0, 3):
    print(f"Cluster {i+1} predicted weight: {wts[i]:.3f} , {wts[i]*100:.3f}%")

Inputs
Week of year : 51
Day of week: 3
Hour: 10
Cluster 1 predicted weight: 0.333 , 33.333%
Cluster 2 predicted weight: 0.333 , 33.333%
Cluster 3 predicted weight: 0.333 , 33.333%
